# BASIL on the open-source models (Google Colab)

Runs the same BASIL descriptive-metric calls as `notebooks/basil_metrics.ipynb` (via `scripts/run_basil_metrics.py`) on six local Ollama models.

- **Posts and text:** the 1000 shortlisted posts in `basil_evaluation/base_dataset.csv`, using the `original_without_both` text as the situation.
- **Per post:** 3 calls, one per condition: abstract (no belief sentence), third party ("{name} believes that I am not the asshole.") and user ("I believe that I am not the asshole."). Each asks "What is the probability that the asker is not the asshole?" and returns JSON `{"probability": float}`. One prompt per call, greedy decoding (temperature 0).
- **Models:** `llama2:7b`, `llama3.1:8b`, `qwen:7b`, `qwen3.5:9b`, `gemma:7b`, `gemma4:e4b`.

**Before you start:** use a GPU runtime (Runtime > Change runtime type > T4 GPU or better), and have `basil_colab_bundle.zip` ready to upload.

**Time:** one model is 3,000 calls for 1000 posts. Results are saved to Google Drive as they arrive, and re-running the run cell resumes where it stopped. Use `N_POSTS` to do a smaller subset first.

## 1. Install Ollama and start the server

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os, subprocess, time, requests

os.environ["OLLAMA_NUM_PARALLEL"] = "4"  # must match WORKERS below
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /content/ollama.log")

## 2. Upload the project files and mount Drive

Upload `basil_colab_bundle.zip`, a zip of these repo files, keeping their folders: `scripts/run_basil_metrics.py`, `run_models.py`, `values.py`, `settings.py`, `utilities/` and `basil_evaluation/base_dataset.csv`. Results are written to Drive so they survive a disconnect.

In [ ]:
!pip -q install pandas tqdm requests anthropic openai

from google.colab import files, drive

if not os.path.exists("/content/basil/scripts/run_basil_metrics.py"):
    uploaded = files.upload()  # choose basil_colab_bundle.zip
    zip_name = next(iter(uploaded))
    !mkdir -p /content/basil && unzip -q -o "{zip_name}" -d /content/basil

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/basil_results_open_models"
os.makedirs(OUT_DIR, exist_ok=True)
%cd /content/basil
!ls

## 3. Settings

In [ ]:
MODELS = ["llama3.1:8b", "qwen3.5:9b", "gemma4:e4b", "qwen:7b", "gemma:7b", "llama2:7b"]

N_POSTS = 20        # posts to run per model; set to None for all 1000 (3,000 calls per model)
WORKERS = 4         # concurrent requests; keep equal to OLLAMA_NUM_PARALLEL above
MAX_TOKENS = 64     # the answer is a small JSON object with one number
NUM_CTX = 4096      # llama2 supports at most 4096

chunk_args = f"--chunk-size {N_POSTS} --chunk 0" if N_POSTS else ""
print(len(MODELS), "models;", N_POSTS or 1000, "posts each;", (N_POSTS or 1000) * 3, "calls per model")

## 4. Smoke test (1 post, first model)

Check that the answers parse and no calls fail before the long run.

In [ ]:
m = MODELS[0]
!ollama pull {m}
!python scripts/run_basil_metrics.py --provider ollama --model {m} --workers {WORKERS} --max-tokens {MAX_TOKENS} --num-ctx {NUM_CTX} --chunk-size 1 --chunk 0 --out-dir /content/smoke_test

## 5. Run all models

One model at a time (each is pulled, run, then unloaded to free GPU memory). Safe to stop and re-run: finished calls are skipped.

In [ ]:
for m in MODELS:
    print(f"\n################ {m} ################")
    !ollama pull {m}
    !python scripts/run_basil_metrics.py --provider ollama --model {m} --workers {WORKERS} --max-tokens {MAX_TOKENS} --num-ctx {NUM_CTX} {chunk_args} --out-dir {OUT_DIR}
    !ollama stop {m}

## 6. Check the results

Look for few request errors. The probabilities are saved exactly as the model returned them (not clipped); the `.csv` per model has one row per post with the abstract, third_party and user columns.

In [ ]:
import glob
import pandas as pd

rows = []
for f in sorted(glob.glob(f"{OUT_DIR}/*__basil.jsonl")):
    d = pd.read_json(f, lines=True)
    ok = d["probability"].dropna()
    rows.append({
        "file": os.path.basename(f), "calls": len(d), "request_errors": int(d["error"].notna().sum()),
        "outside_0_1_%": round(100 * ((ok < 0) | (ok > 1)).mean(), 1),
        "avg_seconds": round(d["seconds"].mean(), 1),
    })
pd.DataFrame(rows)